# Drift Monitoring and Model Decay Lab


## Purpose

This lab introduces model drift.

A model that performs well at launch can degrade when deployment data diverges from training data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
n = 5000

reference = rng.normal(loc=0.0, scale=1.0, size=n)
mild_drift = rng.normal(loc=0.35, scale=1.05, size=n)
severe_drift = rng.normal(loc=1.00, scale=1.25, size=n)

distributions = pd.DataFrame({
    "reference": reference,
    "mild_drift": mild_drift,
    "severe_drift": severe_drift,
})

distributions.describe()

## Population Stability Index

Population Stability Index is a simple monitoring signal for distribution shift. It should not be used as a complete drift framework, but it is useful as a dashboard indicator.

In [ ]:
def population_stability_index(reference_values, current_values, bins=10):
    reference_counts, bin_edges = np.histogram(reference_values, bins=bins)
    current_counts, _ = np.histogram(current_values, bins=bin_edges)

    reference_pct = reference_counts / max(reference_counts.sum(), 1)
    current_pct = current_counts / max(current_counts.sum(), 1)

    epsilon = 1e-6
    psi = np.sum(
        (current_pct - reference_pct)
        * np.log((current_pct + epsilon) / (reference_pct + epsilon))
    )

    return psi

drift_summary = pd.DataFrame([
    {
        "comparison": "reference_vs_mild_drift",
        "psi": population_stability_index(reference, mild_drift),
    },
    {
        "comparison": "reference_vs_severe_drift",
        "psi": population_stability_index(reference, severe_drift),
    },
])

drift_summary

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(reference, bins=40, alpha=0.5, label="reference")
plt.hist(mild_drift, bins=40, alpha=0.5, label="mild drift")
plt.hist(severe_drift, bins=40, alpha=0.5, label="severe drift")
plt.legend()
plt.title("Synthetic Distribution Shift")
plt.xlabel("Feature value")
plt.ylabel("Count")
plt.show()

## Governance Questions

- What level of drift requires review?
- Who receives the alert?
- Is retraining automatic or governed?
- What data is used for retraining?
- How are model versions documented?